# 📙 Bài 4 — Làm sạch dữ liệu

**Tuần 8**

> Đây là kỹ năng chiếm **60–80% thời gian thực tế** của người làm dữ liệu. Không hào nhoáng, nhưng model tốt nhất trên dữ liệu bẩn luôn thua model tầm thường trên dữ liệu sạch.

Chúng ta sẽ làm sạch `data/ban_hang.csv` từ đầu đến cuối, theo đúng thứ tự nên làm.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 120)

DATA = Path("../../../data")
if not (DATA / "ban_hang.csv").exists():
    DATA = Path("data")

tho = pd.read_csv(DATA / "ban_hang.csv")
khach = pd.read_csv(DATA / "khach_hang.csv")
print("Đơn hàng thô:", tho.shape, " | Khách hàng:", khach.shape)

## Quy trình 7 bước

Thứ tự này **không tuỳ tiện** — mỗi bước dựa trên kết quả bước trước:

```
① Khảo sát        → biết mình đang đối mặt với gì
② Sửa kiểu        → phải có kiểu đúng mới lọc/tính được
③ Chuẩn hoá chữ   → gộp các biến thể viết khác nhau
④ Trùng lặp       → xoá trước khi thống kê, kẻo đếm nhân đôi
⑤ Giá trị vô lý   → đánh dấu, chưa xoá vội
⑥ Giá trị thiếu   → điền hoặc loại, có căn cứ
⑦ Kiểm tra lại    → xác nhận đã sạch, ghi lại đã làm gì
```

> Người mới hay nhảy thẳng vào bước ⑥ (điền giá trị thiếu). Làm vậy trước khi sửa kiểu dữ liệu là điền vào một cột mà bạn còn chưa hiểu.

## ① Khảo sát — biết mình đang đối mặt với gì

In [ ]:
def bao_cao_chat_luong(df):
    return (
        pd.DataFrame({
            "cot": df.columns,
            "kieu": [str(k) for k in df.dtypes],
            "so_thieu": df.isna().sum().to_numpy(),
            "ty_le_thieu": df.isna().mean().round(4).to_numpy(),
            "so_gia_tri": df.nunique().to_numpy(),
        })
        .sort_values("ty_le_thieu", ascending=False)
        .reset_index(drop=True)
    )


bao_cao_chat_luong(tho)

**Đọc bảng này — bốn dấu hiệu:**

| Dấu hiệu | Nghĩa | Ở đây |
|---|---|---|
| `kieu = object` ở cột lẽ ra là **số** | Bị lưu dạng chuỗi, hoặc có rác lẫn vào | `don_gia` ⚠️ |
| `ty_le_thieu > 0.5` | Cân nhắc xoá cột, hoặc tìm hiểu vì sao | — |
| `so_gia_tri == 1` | Cột **hằng số**, vô dụng cho model | — |
| `so_gia_tri == số dòng` | Cột **ID** — ghép bảng được, không đưa vào model | `don_id` |

In [ ]:
print("Trùng lặp hoàn toàn:", tho.duplicated().sum())
print("don_id bị trùng     :", tho["don_id"].duplicated().sum())
print()
print("Vài giá trị don_gia (chú ý dạng chuỗi):")
print(tho["don_gia"].astype(str).sample(8, random_state=1).tolist())
print()
print("Vài giá trị ngay_dat (BA định dạng khác nhau!):")
print(tho["ngay_dat"].sample(8, random_state=3).tolist())

## ② Sửa kiểu dữ liệu

In [ ]:
df = tho.copy()

# --- Cột số bị lưu dạng chuỗi
df["don_gia"] = pd.to_numeric(
    df["don_gia"].astype(str).str.strip().str.replace(".", "", regex=False),
    errors="coerce",
)
df["so_luong"] = pd.to_numeric(df["so_luong"], errors="coerce")

print(f"don_gia  — không đổi được: {df['don_gia'].isna().sum()} ô")
print(f"so_luong — không đổi được: {df['so_luong'].isna().sum()} ô")
df[["so_luong", "don_gia"]].describe().round(0)

`errors="coerce"` là tham số quan trọng nhất ở đây: giá trị không đổi được thành `NaN` thay vì **ném lỗi và dừng cả chương trình**. Với dữ liệu thật, bạn muốn xử lý xong rồi *đếm* xem có bao nhiêu ô hỏng.

> ⚠️ **Cẩn thận với dấu chấm!** Ở đây ta bỏ hết dấu chấm vì dữ liệu Việt Nam dùng chấm làm phân cách hàng nghìn (`"1.200.000"` = một triệu hai). Nhưng nếu dữ liệu dùng chấm làm **dấu thập phân** kiểu Anh–Mỹ (`"1200.50"`), bỏ dấu chấm làm sai kết quả **100 lần**.
>
> **Luôn nhìn dữ liệu thật trước khi quyết định.** Đây là lỗi làm sạch nguy hiểm nhất: nó không báo lỗi, chỉ làm mọi con số sai lệch âm thầm.

In [ ]:
# --- Cột ngày có BA định dạng cùng tồn tại
def doi_ngay(s):
    dinh_dang = ["%Y-%m-%d", "%d/%m/%Y", "%m-%d-%Y"]
    kq = pd.to_datetime(s, format=dinh_dang[0], errors="coerce")
    for fmt in dinh_dang[1:]:
        thieu = kq.isna()
        if not thieu.any():
            break
        kq[thieu] = pd.to_datetime(s[thieu], format=fmt, errors="coerce")
    return kq


df["ngay_dat"] = doi_ngay(df["ngay_dat"])
print(f"Đổi được {df['ngay_dat'].notna().sum()} / {len(df)} dòng")
print(f"Khoảng thời gian: {df['ngay_dat'].min().date()} → {df['ngay_dat'].max().date()}")

**Vì sao phải chỉ rõ từng `format` thay vì để pandas tự đoán?**

Vì `"03/04/2025"` là ngày 3 tháng 4, hay tháng 3 ngày 4? Pandas sẽ **đoán**, và có thể đoán khác nhau cho từng dòng. Kết quả: báo cáo doanh thu theo tháng sai lệch mà không có cảnh báo nào.

Chỉ rõ format thì bạn **kiểm soát** được chuyện gì xảy ra.

In [ ]:
# Kho báu mở ra sau khi cột đã là datetime
df["thang"] = df["ngay_dat"].dt.month
df["thu"] = df["ngay_dat"].dt.dayofweek        # 0 = thứ Hai
df["quy"] = df["ngay_dat"].dt.quarter

print("Số đơn theo thứ trong tuần (0=T2 ... 6=CN):")
print(df["thu"].value_counts().sort_index())

## ③ Chuẩn hoá giá trị hạng mục

In [ ]:
print("Trước khi chuẩn hoá — bao nhiêu biến thể của cùng một thành phố:")
print(khach["thanh_pho"].value_counts())

In [ ]:
BANG = {
    "hn": "Ha Noi", "ha noi": "Ha Noi", "hà nội": "Ha Noi",
    "hcm": "TP HCM", "tphcm": "TP HCM", "tp hcm": "TP HCM",
    "tp.hcm": "TP HCM", "ho chi minh": "TP HCM", "sg": "TP HCM",
    "dn": "Da Nang", "da nang": "Da Nang", "đà nẵng": "Da Nang",
    "ct": "Can Tho", "can tho": "Can Tho",
    "hp": "Hai Phong", "hai phong": "Hai Phong",
}

k = khach.copy()
thieu_goc = k["thanh_pho"].isna()

k["thanh_pho"] = (
    k["thanh_pho"].astype(str).str.strip().str.lower().map(BANG).fillna("Khac")
)
k.loc[thieu_goc, "thanh_pho"] = "Khong ro"

print("Sau khi chuẩn hoá:")
print(k["thanh_pho"].value_counts())

**Ba điểm kỹ thuật:**

1. **Ghi nhớ `thieu_goc` TRƯỚC.** Sau `.map()` có hai loại `NaN` trộn lẫn: `NaN` vốn có trong dữ liệu (→ `"Khong ro"`) và `NaN` do map sinh ra vì không có trong bảng (→ `"Khac"`). Không ghi nhớ trước thì mất khả năng phân biệt.

2. **`.map()` vs `.replace()`:** `.map()` biến giá trị lạ thành `NaN` (giúp bạn *phát hiện* chúng), `.replace()` giữ nguyên (giấu chúng đi).

3. **Đây là vòng lặp, không phải làm một lần:** chạy → `value_counts()` lại → nếu `"Khac"` vẫn nhiều thì còn biến thể chưa bắt hết → bổ sung bảng → chạy lại.

In [ ]:
# Chuẩn hoá tên người
print("Trước:", khach["ho_ten"].head(5).tolist())
k["ho_ten"] = k["ho_ten"].str.strip().str.title()
print("Sau  :", k["ho_ten"].head(5).tolist())

## ④ Trùng lặp

In [ ]:
print("Trùng toàn bộ cột :", df.duplicated().sum())
print("Trùng theo don_id :", df["don_id"].duplicated().sum())

# NHÌN các dòng trùng trước khi xoá — keep=False hiện CẢ NHÓM
trung = df[df.duplicated(keep=False)].sort_values("don_id")
print(f"\n{len(trung)} dòng thuộc các nhóm trùng lặp. Vài ví dụ:")
print(trung[["don_id", "danh_muc", "so_luong", "don_gia"]].head(6))

In [ ]:
truoc = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Đã xoá {truoc - len(df)} dòng trùng ({(truoc-len(df))/truoc:.1%})")

**Hai kiểu trùng lặp khác nhau hoàn toàn:**

| Kiểu | Ý nghĩa | Xử lý |
|---|---|---|
| Trùng **toàn bộ** cột | Gần như chắc chắn lỗi kỹ thuật: import hai lần, gọi API bị lặp | Xoá được yên tâm |
| Trùng theo **khoá nghiệp vụ** (`don_id`) | Hai dòng cùng ID nhưng khác nội dung — bản ghi bị sửa? Hai sản phẩm trong cùng đơn? | **Phải nhìn trước khi xoá** |

`keep="first"` giữ bản đầu · `keep="last"` giữ bản cuối (khi bản sau là bản đã cập nhật) · `keep=False` xoá hết mọi bản trùng.

## ⑤ Giá trị vô lý — đánh dấu, chưa xoá

In [ ]:
print("Số lượng âm     :", (df["so_luong"] < 0).sum())
print("Số lượng = 0    :", (df["so_luong"] == 0).sum())
print("Đơn giá = 0     :", (df["don_gia"] == 0).sum())
print("Đơn giá âm      :", (df["don_gia"] < 0).sum())
print()
print("Tuổi vô lý trong bảng khách:")
print(k.loc[(k["tuoi"] < 16) | (k["tuoi"] > 100), "tuoi"].tolist())

> ⚠️ **Số lượng âm có phải luôn là lỗi?** Không! Trong nhiều hệ thống, số lượng âm nghĩa là **hoàn trả hàng**. Xoá đi sẽ làm báo cáo doanh thu bị thổi phồng.
>
> Trước khi xoá, **phải hỏi người hiểu nghiệp vụ**. Đây là lý do bước này chỉ *đánh dấu*, chưa xoá.

Ở đây ta biết chắc đây là dữ liệu sinh ra để tập, nên xử lý như lỗi:

In [ ]:
# Tuổi vô lý -> NaN (giữ lại dòng, chỉ bỏ giá trị sai)
k["tuoi"] = k["tuoi"].where(k["tuoi"].between(16, 100), np.nan)
print("Tuổi sau khi làm sạch:")
print(k["tuoi"].describe().round(1))

# Đơn hàng không hợp lệ -> loại dòng
truoc = len(df)
df = df[
    df["so_luong"].notna() & (df["so_luong"] > 0)
    & df["don_gia"].notna() & (df["don_gia"] > 0)
].reset_index(drop=True)
print(f"\nĐã loại {truoc - len(df)} dòng không hợp lệ ({(truoc-len(df))/truoc:.1%})")

> 📌 **Luôn ghi lại số dòng đã loại và tỷ lệ.** Nếu bạn mất 40% dữ liệu mà không biết, mọi kết luận phía sau đều dựa trên một mẫu đã bị thiên lệch. Trong báo cáo EDA (project P1), đây là thông tin **bắt buộc** phải có.

## ⑥ Giá trị thiếu

In [ ]:
print("Còn thiếu ở bảng khách hàng:")
print(k.isna().sum()[k.isna().sum() > 0])

In [ ]:
# Email thiếu: KHÔNG điền bừa. Sự thiếu vắng CHÍNH LÀ thông tin.
k["co_email"] = k["email"].notna() & (k["email"] != "")
k["email"] = k["email"].fillna("(chưa có)")

print("Tỷ lệ khách có email:", f"{k['co_email'].mean():.1%}")
print()
print("Tuổi trung bình theo nhóm có/không có email:")
print(k.groupby("co_email")["tuoi"].mean().round(1))

> 💡 **Đây là ý tưởng quan trọng nhất của bước ⑥.** Trước khi điền, hãy tự hỏi: *sự thiếu vắng này có mang thông tin không?*
>
> Nếu khách không khai email vì họ mua ở cửa hàng chứ không mua online, thì cột `co_email` **dự đoán được hành vi** — có giá trị hơn cả cột `email` gốc. Điền bừa là vứt thông tin đó đi.

In [ ]:
# Tuổi thiếu: điền bằng median THEO NHÓM (thành phố), không phải median toàn cục
median_nhom = k.groupby("thanh_pho")["tuoi"].transform("median")

k["tuoi_da_dien"] = (
    k["tuoi"].fillna(median_nhom).fillna(k["tuoi"].median()).fillna(0)
)

print("So sánh hai cách điền:")
print(f"  Median toàn cục : {k['tuoi'].median():.1f}")
print("  Median theo thành phố:")
print(k.groupby("thanh_pho")["tuoi"].median().round(1).to_string())

**Ba tầng dự phòng** — từ chính xác nhất đến an toàn nhất:

```
① median của NHÓM      → chính xác nhất
② median TOÀN CỤC      → khi cả nhóm đều thiếu
③ số 0                 → khi toàn bộ cột đều thiếu
```

Thiếu tầng ① thì bạn điền một con số vô nghĩa; thiếu tầng ②③ thì vẫn còn `NaN` sót lại và lỗi ở bước sau.

**Ví dụ vì sao điền theo nhóm quan trọng:** giá một chiếc *laptop* bị thiếu mà điền bằng median của cả bảng (gồm cả phụ kiện 50 nghìn) thì ra con số vô lý. Điền bằng median riêng của nhóm laptop mới hợp lý.

> ⚠️ Dù điền khéo đến mấy, bạn vẫn đang **bịa dữ liệu**. Nếu một cột thiếu 60%, điền vào không làm nó thành thật. Lúc đó nên xoá hẳn cột, hoặc giữ cột đánh dấu như cách làm với email ở trên.

## ⑦ Kiểm tra lại & ghi chép

In [ ]:
df["thanh_tien"] = (df["so_luong"] * df["don_gia"] * (1 - df["giam_gia"])).round(0)

print("=" * 58)
print("  BÁO CÁO LÀM SẠCH")
print("=" * 58)
print(f"  Dòng ban đầu      : {len(tho):>6}")
print(f"  Dòng còn lại      : {len(df):>6}  ({len(df)/len(tho):.1%})")
print(f"  Đã loại           : {len(tho)-len(df):>6}")
print()
print("  Kiểm tra cuối:")
print(f"    Còn giá trị thiếu?    {df[['so_luong','don_gia','thanh_tien']].isna().sum().sum()}")
print(f"    Còn trùng lặp?        {df.duplicated().sum()}")
print(f"    Còn giá trị âm?       {(df['thanh_tien'] < 0).sum()}")
print(f"    Kiểu ngày đúng?       {pd.api.types.is_datetime64_any_dtype(df['ngay_dat'])}")
print()
print(f"  Tổng doanh thu    : {df['thanh_tien'].sum():>18,.0f} VND")
print("=" * 58)

In [ ]:
# Ngoại lai: xem xét, KHÔNG xoá vội
q1, q3 = df["thanh_tien"].quantile([0.25, 0.75])
iqr = q3 - q1
nl = df[df["thanh_tien"] > q3 + 1.5 * iqr]

print(f"{len(nl)} đơn vượt ngưỡng IQR ({len(nl)/len(df):.1%})")
print(f"Chiếm {nl['thanh_tien'].sum()/df['thanh_tien'].sum():.1%} tổng doanh thu")
print()
print("Top 5 đơn lớn nhất:")
print(df.nlargest(5, "thanh_tien")[["don_id", "danh_muc", "so_luong", "thanh_tien"]])

Nhìn kỹ: các đơn lớn nhất là **laptop số lượng 40–120 chiếc**. Đó không phải lỗi nhập liệu — đó là **khách doanh nghiệp**.

> 🔑 Xoá chúng đi là vứt bỏ đúng nhóm khách hàng giá trị nhất. Cách xử lý đúng là **tách riêng để phân tích**, không phải xoá.

In [ ]:
# Lưu lại kết quả để dùng cho project P1
df.to_csv(DATA / "ban_hang_sach.csv", index=False, encoding="utf-8-sig")
k.to_csv(DATA / "khach_hang_sach.csv", index=False, encoding="utf-8-sig")
print("Đã lưu:")
print("  data/ban_hang_sach.csv")
print("  data/khach_hang_sach.csv")

> 📌 `encoding="utf-8-sig"` để **Excel trên Windows** mở đúng tiếng Việt. Dùng `utf-8` thường thì Excel hiển thị chữ loạn — chi tiết nhỏ nhưng làm hỏng ấn tượng khi gửi báo cáo cho người khác.

## Tổng kết — checklist làm sạch

```
① KHẢO SÁT     □ shape  □ head  □ info  □ describe  □ tỷ lệ thiếu  □ trùng lặp
② KIỂU         □ cột số bị lưu dạng chuỗi?   □ cột ngày đã là datetime?
③ CHỮ          □ value_counts từng cột hạng mục  □ chuẩn hoá  □ kiểm tra lại
④ TRÙNG        □ nhìn trước khi xoá  □ ghi lại số dòng đã xoá
⑤ VÔ LÝ        □ âm, 0, cực trị  □ HỎI nghiệp vụ trước khi xoá
⑥ THIẾU        □ hỏi VÌ SAO thiếu  □ cân nhắc cột đánh dấu  □ điền theo nhóm
⑦ KIỂM TRA     □ chạy lại báo cáo  □ ghi rõ đã loại bao nhiêu, vì sao
```

## ✍️ Bài tập

```bash
pytest tests/phase03/test_ex03.py -v
```